# ship pileup at Seattle's East Waterway entrance

run `seattle_shoreline.ipynb` first; this reuses its fgout depths and velocities

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch
from netCDF4 import Dataset
import numpy as np
import pandas as pd
from shapely.geometry import box, shape
from shapely.ops import transform, unary_union
from shapely.prepared import prep
from shapely.strtree import STRtree
from IPython.display import Video, display

HERE = Path.cwd().resolve()
if not (HERE / 'setrun.py').exists(): HERE = HERE / 'examples' / 'seattle_debris'
if not (HERE / 'setrun.py').exists(): raise FileNotFoundError('Run from the Seattle example or repository root')
sys.path.insert(0, str(HERE.parents[1] / 'src'))
from geoclaw_debris.ship_tracking import Flow, Ship, check_pose, hull_polygon, simulate_fleet

FLOW_FILE = HERE / '_output' / 'seattle_shoreline' / 'seattle_fault_fgout.nc'
OUT = HERE / '_output' / 'seattle_harbor_pileup'
if not FLOW_FILE.exists(): raise FileNotFoundError('Run seattle_shoreline.ipynb first to create ' + str(FLOW_FILE))
OUT.mkdir(parents=True, exist_ok=True)

BOUNDS = [-122.350, -122.339, 47.581, 47.595]
START, END = 300., 540.
DT = 2.
counts = [1, 5, 20, 80]

# read the saved Seattle Fault tsunami frames for the southward pulse
inventory_file = HERE.parents[1] / 'SeattleBuildings' / 'Seattle_BuildingInventory.geojson'
if not inventory_file.exists(): raise FileNotFoundError('Missing building inventory: ' + str(inventory_file))
features = json.loads(inventory_file.read_text())['features']
with Dataset(FLOW_FILE) as ds:
    assert ds.inventory_sha256 == hashlib.sha256(inventory_file.read_bytes()).hexdigest(), 'Building inventory changed; rerun seattle_shoreline.ipynb'
    times = np.asarray(ds['time'][:])
    assert START in times and END in times, 'The FGout record must include both window endpoints'
    frames = (times >= START) & (times <= END)
    depth = np.asarray(ds['h'][frames])
    u, v = np.asarray(ds['u'][frames]), np.asarray(ds['v'][frames])
    flow = Flow(dict(lon=ds['lon'][:], lat=ds['lat'][:], time=times[frames], h=depth, u=u, v=v))
    initial_depth = np.asarray(ds['h'][0])
    forcing = ds.forcing

domain = box(flow.lon.min(), flow.lat.min(), flow.lon.max(), flow.lat.max())
footprints = [shape(f['geometry']) for f in features]
footprints = [p for p in footprints if p.intersects(domain)]
buildings = unary_union(footprints)
obstacles = prep(transform(flow.to_xy, buildings))
view = box(BOUNDS[0], BOUNDS[2], BOUNDS[1], BOUNDS[3])
metric_footprints = [transform(flow.to_xy, p) for p in footprints if p.intersects(view)]
gx, gy = flow.to_xy(flow.lon, flow.lat)

# place the fleet north of the East Waterway entrance
ship = Ship(length_m=30., beam_m=8., draft_m=1.5)
HEADING = -90.
GATE = [-122.3462, -122.34255, 47.5895]
gate_x, gate_y = flow.to_xy(np.array(GATE[:2]), GATE[2])
# pack the approach in rows, with the front row just north of the entrance
x, y = flow.to_xy(-122.3445, 47.58978)
lanes = [0, -1, 1, -2, 2, -3, 3, -4, 4]
centers = [(x+12.*lane, y+34.*row) for row in range(16) for lane in lanes]
starts, starting_hulls = [], []
for center in centers:
    clear, _, _ = check_pose(flow, obstacles, ship, center, np.deg2rad(HEADING), START)
    if not clear: continue
    starts.append(flow.to_lonlat(*center))
    starting_hulls.append(hull_polygon(center, np.deg2rad(HEADING), ship))
    if len(starts) == max(counts): break
assert len(starts) == max(counts), 'The approach grid must fit the full fleet'
assert all(hull.bounds[1] > gate_y for hull in starting_hulls)
assert all(not starting_hulls[i].intersects(starting_hulls[j]) for i in range(len(starts)) for j in range(i))
print(forcing)

In [ ]:
def harbor_map(ax):
    ax.contourf(gx, gy, initial_depth.T, levels=[-.01, .1, float(initial_depth.max())+1], colors=['#e5e0d6', '#dceaf0'])
    for footprint in metric_footprints:
        for piece in getattr(footprint, 'geoms', [footprint]):
            if piece.geom_type == 'Polygon':
                ax.add_patch(PolygonPatch(np.asarray(piece.exterior.coords), facecolor='#827e76', edgecolor='none', alpha=.7))
    ax.plot(gate_x, [gate_y, gate_y], '--', color='#222', lw=1)
    x, y = flow.to_xy(np.array(BOUNDS[:2]), np.array(BOUNDS[2:]))
    ax.set(xlim=x, ylim=y, xlabel='east (m)', ylabel='north (m)', aspect='equal')

def outline(row):
    return hull_polygon((row['x_m'], row['y_m']), np.deg2rad(row['heading_deg']), ship)

def entry_time(path):
    for a, b in zip(path, path[1:]):
        if a['y_m'] >= gate_y > b['y_m']:
            fraction = (gate_y-a['y_m'])/(b['y_m']-a['y_m'])
            x = a['x_m'] + fraction*(b['x_m']-a['x_m'])
            if gate_x[0] <= x <= gate_x[1]:
                return a['time_s'] + fraction*(b['time_s']-a['time_s'])
    return np.nan

def case_label(count, dt, collisions):
    return f'{count} ships, {dt:g} s step' + ('' if collisions else ', collisions off')

In [ ]:
cases = [(count, DT, True) for count in counts] + [(max(counts), DT/2, True), (max(counts), DT, False)]
runs, entries, contacts, waiting, summary = {}, {}, {}, {}, []
for count, dt, collisions in cases:
    case = count, dt, collisions
    paths = simulate_fleet(flow, buildings, [ship]*count, starts[:count],
                           heading_deg=HEADING, start_time=START, end_time=END, dt=dt, collisions=collisions)
    overlap_steps = 0
    max_overlap_m2 = 0.
    for rows in zip(*paths):
        outlines = [outline(r) for r in rows]
        assert all(not obstacles.intersects(hull) for hull in outlines)
        i, j = STRtree(outlines).query(outlines, predicate='intersects')
        area = max((outlines[a].intersection(outlines[b]).area for a, b in zip(i, j) if a < b), default=0.)
        overlap_steps += area > 1.e-6
        max_overlap_m2 = max(max_overlap_m2, area)

    runs[case] = paths
    entries[case] = np.array([entry_time(path) for path in paths])
    times = np.array([r['time_s'] for r in paths[0]])
    contacts[case] = np.array([sum(r['status'] == 'ship contact' for r in rows) for rows in zip(*paths)])
    waiting[case] = np.array([sum(r['y_m'] >= gate_y for r in rows) for rows in zip(*paths)])
    entry_times = entries[case][np.isfinite(entries[case])]
    summary.append(dict(ships=count, dt_s=dt, collisions=collisions, entered=len(entry_times),
                        first_contact_s=min(times[contacts[case] > 0], default=np.nan),
                        peak_ships_in_contact=int(contacts[case].max()),
                        last_entry_s=max(entry_times, default=np.nan),
                        waiting_at_end=int(waiting[case][-1]), ships_in_contact_at_end=int(contacts[case][-1]),
                        overlap_steps=overlap_steps, max_overlap_m2=max_overlap_m2))

tracks = pd.concat([pd.DataFrame(path).assign(ships=count, dt_s=dt, collisions=collisions, ship=k)
                    for (count, dt, collisions), paths in runs.items() for k, path in enumerate(paths)], ignore_index=True)
assert np.isfinite(tracks[['x_m', 'y_m', 'heading_deg', 'u_mps', 'v_mps']]).all().all()
assert np.isfinite(entries[1, DT, True]).all(), 'The single ship must enter the waterway'
tracks.to_csv(OUT / 'pileup_paths.csv', index=False)
summary = pd.DataFrame(summary)
summary.to_csv(OUT / 'pileup_summary.csv', index=False)
display(summary.round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 11), constrained_layout=True)
for ax, case in zip(axes.flat, cases[:len(counts)]):
    harbor_map(ax)
    for path, entered in zip(runs[case], np.isfinite(entries[case])):
        p = pd.DataFrame(path)
        color = '#36576b' if entered else '#d84b45'
        ax.plot(p.x_m, p.y_m, color=color, lw=1)
        ax.plot(p.x_m.iloc[0], p.y_m.iloc[0], 'o', color=color, ms=3)
        hull = outline(path[-1])
        ax.add_patch(PolygonPatch(np.asarray(hull.exterior.coords), facecolor=color, edgecolor='#222'))
    ax.set_title(case_label(*case))
fig.savefig(OUT / 'trajectories.png', dpi=150)
plt.show()

fig, (ax, waiting_ax, contact_ax) = plt.subplots(3, 1, figsize=(8, 8), sharex=True, constrained_layout=True)
for count, dt, collisions in cases:
    case = count, dt, collisions
    times = np.array([r['time_s'] for r in runs[case][0]])
    entered = (entries[case][:, None] <= times).sum(axis=0)
    line, = ax.step(times, entered, where='post', label=case_label(*case), ls='-' if collisions else '--')
    if count == max(counts):
        waiting_ax.step(times, waiting[case], where='post', color=line.get_color(), ls=line.get_linestyle())
    if collisions:
        contact_ax.step(times, contacts[case], where='post', color=line.get_color())
ax.set(ylabel='ships entered')
ax.legend()
waiting_ax.set(ylabel='80 ship runs\nships north of entrance')
contact_ax.set(xlabel='time after earthquake (s)', ylabel='ships in contact')
fig.savefig(OUT / 'throughput.png', dpi=150)
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation, FFMpegWriter
import matplotlib as mpl
import shutil

case = max(counts), DT, True
paths = runs[case]
ship_times = np.array([r['time_s'] for r in paths[0]])
fig, ax = plt.subplots(figsize=(7, 8), constrained_layout=True)
harbor_map(ax)
water = ax.pcolormesh(gx, gy, np.ma.masked_less(depth[0].T, .1), cmap='Blues', vmin=0, vmax=20, shading='auto', alpha=.6)
fig.colorbar(water, ax=ax, label='water depth (m)', shrink=.65)
arrows = ax.quiver(gx[::3], gy[::3], u[0, ::3, ::3].T, v[0, ::3, ::3].T, angles='xy', scale_units='xy', scale=.06, color='#36576b')
ax.quiverkey(arrows, .08, .96, 2., '2 m/s', labelpos='E')
hulls, trails = [], []
for path in paths:
    hull = PolygonPatch(np.zeros((4, 2)), facecolor='#36576b', edgecolor='#222', zorder=5)
    ax.add_patch(hull)
    hulls.append(hull)
    trails.append(ax.plot([], [], color='#36576b', lw=1, alpha=.4, zorder=4)[0])
xy = [np.array([[r['x_m'], r['y_m']] for r in path]) for path in paths]

def update(frame):
    time = flow.times[frame]
    step = np.searchsorted(ship_times, time, side='right')-1
    water.set_array(np.ma.masked_less(depth[frame].T, .1).ravel())
    wet = depth[frame, ::3, ::3].T >= .1
    arrows.set_UVC(np.ma.masked_where(~wet, u[frame, ::3, ::3].T),
                   np.ma.masked_where(~wet, v[frame, ::3, ::3].T))
    for k, path in enumerate(paths):
        hulls[k].set_xy(np.asarray(outline(path[step]).exterior.coords))
        contact = path[step]['status'] == 'ship contact'
        hulls[k].set_edgecolor('#ff8c00' if contact else '#222')
        hulls[k].set_linewidth(2. if contact else 1.)
        trails[k].set_data(xy[k][:step+1, 0], xy[k][:step+1, 1])
    entered = (entries[case] <= time).sum()
    ax.set_title(f'East Waterway: {time:.0f} s; {entered} entered; {waiting[case][step]} north; {contacts[case][step]} in contact')

animation = FuncAnimation(fig, update, frames=len(flow.times), interval=125)
plt.close(fig)
ffmpeg = shutil.which('ffmpeg')
if ffmpeg is None:
    local_ffmpeg = HERE.parents[1] / '.venv' / 'bin' / 'ffmpeg'
    if local_ffmpeg.is_file(): ffmpeg = str(local_ffmpeg)
if ffmpeg is None: raise RuntimeError('Install ffmpeg to export the animation')
movie = OUT / 'seattle_harbor_pileup.mp4'
with mpl.rc_context({'animation.ffmpeg_path': ffmpeg}):
    animation.save(str(movie), writer=FFMpegWriter(fps=8), dpi=100)
Video(str(movie), embed=True)